# Lab 8 — Neural PDEs and deep hedging
**Learning to Replicate Risks — MScT MaQi (X–ENSAE), Part II**

Companion notebook to Chapter 4 of the lecture notes. Fill in every `TODO`.

**House rule.** Every numerical claim must come with an error bar over at least 5 seeds.
A single run is not a result.

| Part | Points | Content |
|---|---|---|
| A | 5 | PINN for Black–Scholes, then a basket |
| B | 6 | Deep BSDE vs. deep backward DP (DBDP) |
| C | 7 | Deep hedging with transaction costs |
| Quality | 2 | seeds, figures, honest discussion |

In [ ]:
import math, time
import numpy as np
import torch, torch.nn as nn
import matplotlib.pyplot as plt
from scipy.stats import norm

torch.set_default_dtype(torch.float64)          # PDE residuals need the precision
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(s):
    np.random.seed(s); torch.manual_seed(s)

class MLP(nn.Module):
    """Smooth MLP. tanh (not ReLU): we need SECOND derivatives."""
    def __init__(self, d_in, d_out=1, width=32, depth=3, act=nn.Tanh):
        super().__init__()
        layers, d = [], d_in
        for _ in range(depth):
            layers += [nn.Linear(d, width), act()]; d = width
        layers += [nn.Linear(d, d_out)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)

def d(y, x, create_graph=True):
    """Gradient of a scalar-valued batch output y wrt x."""
    return torch.autograd.grad(y, x, torch.ones_like(y), create_graph=create_graph)[0]

---
## Part A — A physics-informed network for Black–Scholes

### A.1 One dimension, against the closed form

$$\partial_t u+\tfrac12\sigma^2s^2\partial^2_{ss}u+rs\,\partial_su-ru=0,\qquad u(T,s)=(s-K)^+ .$$

Two variants: plain residual loss, and the **hard-wired terminal condition**
$U_\theta(t,s)=(s-K)^++(T-t)\Phi_\theta(t,s)$.

In [ ]:
SIG, R, T, K = 0.2, 0.02, 1.0, 1.0

def bs_call(s, t, K=K, r=R, sig=SIG, T=T):
    tau = np.maximum(T - t, 1e-12)
    d1 = (np.log(s / K) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))
    d2 = d1 - sig * np.sqrt(tau)
    return s * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)

def bs_delta(s, t, K=K, r=R, sig=SIG, T=T):
    tau = np.maximum(T - t, 1e-12)
    d1 = (np.log(s / K) + (r + 0.5 * sig**2) * tau) / (sig * np.sqrt(tau))
    return norm.cdf(d1)

In [ ]:
class PricePINN(nn.Module):
    """hardwired=True  ->  U(t,s) = payoff(s) + (T-t) * Phi(t,s)."""
    def __init__(self, hardwired=True, width=32, depth=3):
        super().__init__()
        self.body = MLP(2, 1, width, depth)
        self.hardwired = hardwired
    def forward(self, t, s):
        # TODO: feed NORMALISED inputs  (t/T , log(s/K))  -- raw prices train very poorly
        z = torch.cat([t / T, torch.log(s / K)], dim=-1)
        raw = self.body(z)
        if self.hardwired:
            payoff = torch.clamp(s - K, min=0.0)
            return payoff + (T - t) * raw
        return raw

def pinn_loss(net, t, s):
    t.requires_grad_(True); s.requires_grad_(True)
    u    = net(t, s)
    u_s  = d(u, s); u_ss = d(u_s, s); u_t = d(u, t)
    res  = u_t + 0.5 * SIG**2 * s**2 * u_ss + R * s * u_s - R * u
    loss = (res**2).mean()
    if not net.hardwired:
        sT = s.detach().clone()
        tT = torch.full_like(sT, T)
        loss = loss + ((net(tT, sT) - torch.clamp(sT - K, min=0.0))**2).mean()
    return loss

def sample_uniform(n):
    t = torch.rand(n, 1, device=DEV) * T
    s = torch.exp(torch.rand(n, 1, device=DEV) * (math.log(2.0) - math.log(0.5)) + math.log(0.5))
    return t, s

def sample_along_dynamics(n, vol_mult=1.0):
    # TODO A.2: nu_n = law of S_tau under the risk-neutral dynamics
    t = torch.rand(n, 1, device=DEV) * T
    w = torch.randn(n, 1, device=DEV) * math.sqrt(1.0) * torch.sqrt(t)
    sig = SIG * vol_mult
    s = torch.exp((R - 0.5 * sig**2) * t + sig * w)
    return t, s

def train_pinn(hardwired=True, sampler=sample_uniform, iters=4000, seed=0, lr=1e-3, batch=512):
    set_seed(seed)
    net = PricePINN(hardwired).to(DEV)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, iters, eta_min=1e-5)
    hist = []
    for k in range(iters):
        t, s = sampler(batch)
        loss = pinn_loss(net, t, s)
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        hist.append(loss.item())
    return net, hist

In [ ]:
# TODO A.1:
#   - train both variants over >= 5 seeds
#   - relative L2 and max abs error against bs_call on s in [0.6, 1.6] at t = 0 and t = T/2
#   - plot the learnt delta (autograd) against bs_delta
#
# TODO A.2:
#   - repeat with sample_uniform / sample_along_dynamics(1.0) / sample_along_dynamics(2.0)
#   - evaluate on BOTH a uniform and a log-normal test set; write down the rule you infer
#
# TODO A.3 (**):
#   - basket call in dimension n in {1, 5, 20}, correlation 0.3
#   - benchmark: Monte-Carlo with 1e6 paths and its confidence interval
#   - report training time vs n and contrast with the N^n of a grid

---
## Part B — Deep BSDE and deep backward dynamic programming

Test PDE (Hopf–Cole, exact solution available):
$$\partial_t v+\tfrac12\Delta_x v=\tfrac12|\nabla_x v|^2,\quad v(T,\cdot)=g
\qquad\Longrightarrow\qquad v(0,x_0)=-\log \mathbb E\big[e^{-g(x_0+W_T)}\big].$$
Here $f(x,y,z)=-\tfrac12|z|^2$ and $X$ is a standard Brownian motion.

In [ ]:
def g_test(x):                       # x: (B, n)
    return torch.log(0.5 * (1.0 + (x**2).sum(-1, keepdim=True)))

def hopf_cole_benchmark(x0, T_, n, n_paths=2_000_000, seed=0):
    set_seed(seed)
    x = torch.as_tensor(x0, device=DEV).broadcast_to((1, n)) + math.sqrt(T_) * torch.randn(n_paths, n, device=DEV)
    return (-torch.log(torch.exp(-g_test(x)).mean())).item()

def f_driver(x, y, z):               # -1/2 |z|^2
    return -0.5 * (z**2).sum(-1, keepdim=True)

In [ ]:
def deep_bsde(n=1, T_=1.0, N=60, x0=0.0, iters=4000, seed=0, width=None, lr=1e-2, batch=512):
    """Global forward shooting (Han-Jentzen-E). Algorithm 2.2 of the notes."""
    set_seed(seed); h = T_ / N; width = width or (n + 10)
    y0 = nn.Parameter(torch.zeros(1, device=DEV))
    zs = nn.ModuleList([MLP(n, n, width, 2).to(DEV) for _ in range(N)])
    opt = torch.optim.Adam(list(zs.parameters()) + [y0], lr=lr)
    for k in range(iters):
        x = torch.full((batch, n), x0, device=DEV)
        y = y0.expand(batch, 1)
        for i in range(N):
            z  = zs[i](x)
            dw = math.sqrt(h) * torch.randn(batch, n, device=DEV)
            y  = y - f_driver(x, y, z) * h + (z * dw).sum(-1, keepdim=True)
            x  = x + dw
        loss = ((y - g_test(x))**2).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return y0.item()


def dbdp(n=1, T_=1.0, N=60, x0=0.0, iters=400, seed=0, width=None,
         variant=1, transfer=True, lr=1e-3, batch=512):
    """Local backward scheme (Hure-Pham-Warin). Algorithm 2.3 of the notes.
    variant=1 -> separate networks (U, Z);  variant=2 -> Z = grad U by autograd."""
    set_seed(seed); h = T_ / N; width = width or (n + 10)
    # TODO: implement the backward loop
    #   U_{N} = g
    #   for i = N-1 ... 0 :
    #       sample X_{t_i} ~ mu  (here: N(x0, t_i I)), dW, X_{t_{i+1}} = X_{t_i} + dW
    #       minimise | U_{i+1}(X_{i+1}) - U(X_i) + f(X_i,U,Z) h - <Z, dW> |^2
    #       TRANSFER LEARNING: initialise (U,Z) at step i with the parameters from step i+1
    raise NotImplementedError

# TODO B.1: table of v(0,x0) vs Hopf-Cole, mean +/- std over 5 seeds, plus wall-clock time,
#           for n in {1, 5, 20}; plot SGD iterations per time step with and without transfer.
# TODO B.2: fix n = 1, vary T in {0.5, 1, 1.5, 2} with N = 120*T; plot the error vs T for
#           deep BSDE and DBDP1. Does a bigger network rescue deep BSDE? Try it.
# TODO B.3 (**): CVA straddle; no closed form -> cross-validate DBDP against deep BSDE.

---
## Part C — Deep hedging

$$\mathrm{P\&L}_T=-g(S_{t_N})+\sum_i \delta_{t_i}\,(S_{t_{i+1}}-S_{t_i})-\sum_i c(\delta_{t_i}-\delta_{t_{i-1}}),
\qquad \min_\theta\ \rho(p_0+\mathrm{P\&L}_T).$$

**C.1 is the licence to trust C.2 and C.3.** With no costs, the learnt hedge must reproduce
the Black–Scholes delta. Do not proceed until it does.

In [ ]:
def simulate_bs(n_paths, N, S0=1.0, sig=SIG, r=0.0, T_=T, seed=None):
    if seed is not None: set_seed(seed)
    h  = T_ / N
    dw = math.sqrt(h) * torch.randn(n_paths, N, device=DEV)
    logS = (r - 0.5 * sig**2) * h + sig * dw
    S = S0 * torch.exp(torch.cumsum(logS, dim=1))
    return torch.cat([torch.full((n_paths, 1), S0, device=DEV), S], dim=1)   # (B, N+1)


class HedgeNet(nn.Module):
    """One network shared across dates, with time as an input.
    use_prev=True feeds the previous position -- REQUIRED under transaction costs
    (it is what restores the Markov property of the augmented state)."""
    def __init__(self, use_prev=False, width=32, depth=2):
        super().__init__()
        self.use_prev = use_prev
        self.net = MLP(2 + int(use_prev), 1, width, depth)
    def forward(self, t_over_T, logS, prev=None):
        feats = [t_over_T, logS] + ([prev] if self.use_prev else [])
        return self.net(torch.cat(feats, dim=-1))


def pnl(net, S, kappa=0.0, payoff=lambda s: torch.clamp(s - K, min=0.0), T_=T):
    """Non-anticipative by construction: delta_{t_i} multiplies S_{t_{i+1}} - S_{t_i}."""
    B, Np1 = S.shape; N = Np1 - 1
    prev = torch.zeros(B, 1, device=S.device)
    out  = torch.zeros(B, 1, device=S.device)
    for i in range(N):
        s  = S[:, i:i+1]
        ti = torch.full_like(s, i / N)
        dl = net(ti, torch.log(s / K), prev if net.use_prev else None)
        out = out + dl * (S[:, i+1:i+2] - s) - kappa * s * (dl - prev).abs()
        prev = dl
    out = out - payoff(S[:, -1:])
    # liquidate the final position at no extra impact (convention: state it in your report)
    return out


def rho_quadratic(z):  return (z**2).mean()
def rho_entropic(z, beta=1.0):  return torch.logsumexp(-beta * z.squeeze(-1), 0).sub(math.log(z.shape[0])) / beta
def rho_cvar(z, w, alpha=0.95):  return w + torch.clamp(-z - w, min=0.0).mean() / (1 - alpha)


def train_hedge(kappa=0.0, use_prev=False, risk="quad", N=30, iters=3000,
                batch=512, seed=0, lr=1e-3, beta=1.0, alpha=0.95):
    set_seed(seed)
    net = HedgeNet(use_prev=use_prev).to(DEV)
    p0  = nn.Parameter(torch.zeros(1, device=DEV))     # premium, learnt jointly
    w   = nn.Parameter(torch.zeros(1, device=DEV))     # Rockafellar-Uryasev variable
    opt = torch.optim.Adam(list(net.parameters()) + [p0, w], lr=lr)
    for k in range(iters):
        S = simulate_bs(batch, N)
        z = p0 + pnl(net, S, kappa)
        if   risk == "quad": loss = rho_quadratic(z)
        elif risk == "ent":  loss = rho_entropic(z, beta)
        elif risk == "cvar": loss = rho_cvar(z, w, alpha)
        else: raise ValueError(risk)
        opt.zero_grad(); loss.backward(); opt.step()
    return net, p0.item(), w.item()

In [ ]:
# ---- C.1  THE SANITY CHECK -- run this first, and only continue if it passes -------------
net, p0, _ = train_hedge(kappa=0.0, use_prev=False, risk="quad", N=30, iters=8000, seed=0)
# ~8k-15k iterations are needed for the premium to land within ~2% of the BS price;
# the DELTA matches much earlier than the premium does.
# simulate_bs uses r = 0, so the benchmark must also use r = 0 (not the Part A rate R).
print(f"learnt premium p0 = {p0:.4f}   |   Black-Scholes price = {bs_call(1.0, 0.0, r=0.0):.4f}")

grid = torch.linspace(0.6, 1.5, 200, device=DEV).reshape(-1, 1)
plt.figure(figsize=(7, 4))
for tt in [0.0, 0.5, 0.9]:
    with torch.no_grad():
        dl = net(torch.full_like(grid, tt), torch.log(grid / K)).cpu().numpy().ravel()
    plt.plot(grid.cpu().numpy().ravel(), dl, label=f"learnt, t={tt}")
    plt.plot(grid.cpu().numpy().ravel(), bs_delta(grid.cpu().numpy().ravel(), tt, r=0.0), "--",
             label=f"BS delta, t={tt}")
plt.xlabel("S"); plt.ylabel("delta"); plt.legend(fontsize=8)
plt.title("C.1 frictionless sanity check"); plt.tight_layout(); plt.show()

# If the curves do NOT coincide, check in this order:
#   1. inputs not normalised
#   2. off-by-one in the P&L sum (leakage -> suspiciously GOOD P&L)
#   3. too few rebalancing dates
#   4. learning rate too high

In [ ]:
# TODO C.2: kappa in {0, 5, 20, 50} bp, use_prev=True.
#   - plot delta_theta(T/2, S, delta_prev) for two values of delta_prev, overlay the BS delta
#   - measure the width of the no-transaction band vs kappa; compare with the kappa^(1/3) law
#   - risk-cost frontier: rho(P&L) against average cost paid, learnt strategy vs delta hedge
#
# TODO C.3: risk = "ent" (beta in {1,5}) and risk = "cvar" (alpha = 0.95).
#   - overlay the three P&L histograms at kappa = 20 bp
#   - which measure gives the fattest left tail? the widest distribution? why?
#   - what happens to training stability as beta grows?
#
# TODO C.4 (**): basket call on n = 5 correlated assets, hedged with the 5 underlyings.
#   Benchmark: the sum of individual deltas.

---
## What to hand in

A notebook that runs top to bottom and produces, at minimum:

1. **A** — price and delta figures (both PINN variants overlaid on the exact solution) and an error table over 5 seeds; the training-measure comparison.
2. **B** — the accuracy/time table against Hopf–Cole; the error-vs-$T$ figure for deep BSDE and DBDP.
3. **C** — the frictionless check; the no-transaction band; the risk–cost frontier; the three P&L histograms.
4. At most one page of discussion per part. **Report what failed and why.**